In [ ]:
# Note: the RNN used here is
# very much based on Manuel Beiran's code:
# https://github.com/emebeiran/low-ranz2020

import os
import sys

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import torch
from scipy.stats import pearsonr
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..")))

from fig_utils.perturbation import (
    compute_perturbation_distance_stats,
    generate_w_perturb_x,
    perturbation_goal_bounds,
    position_latent_indices,
)
from fig_utils.plots import (
    plot_basis_3d_trajectories,
    plot_connectivity_matrix,
    plot_distance_moved_vs_class_mean,
    plot_perturbation_latent_snapshots_attractor,
    plot_ring_demo_trajectories,
    plot_ring_subspace_streamplot,
    plot_spike_histogram_stats,
    plot_student_teacher_overview,
    plot_unit_activity_by_stimulus,
    plot_unit_trials_overlay,
    compute_ring_subspace_flow,
)
from fig_utils.spike_stats import spike_histogram_stats_from_array
from fig_utils.toy_models import (
    generate_student_teacher_dataset,
    init_ring_attractor_lrrnn,
)
from vi_rnn.data_utils import make_all_trials, stim_end_bins
from vi_rnn.fixed_points import run_scify


%matplotlib inline




In [ ]:
cmap = mpl.colors.ListedColormap(sns.color_palette("husl", n_colors=6))

In [ ]:
save = False  # set True to write .npy files


# --- Ring-attractor toy low-rank RNN parameters ---
relu_bias = 2

g_rec = 3.0
P = 6

alpha_mn = -0.5
alpha_I = 0.75

input_noise = 0.5
sigma_dyn = 0.1  # 05

sigma_b = 0.2
sigma_I = 6

dt = 0.05
tau = 0.2
N = 600

seed = 0

noise_scale = 1.0

# Initialize the toy model ONCE (provides M, N, I, J and population means)
rnn = init_ring_attractor_lrrnn(
    seed=seed,
    P=P,
    N=N,
    dt=dt,
    tau=tau,
    relu_bias=relu_bias,
    g_rec=g_rec,
    sigma_pat=sigma_b,
    alpha_mn=alpha_mn,
    alpha_I=alpha_I,
    sigma_I=sigma_I,
    sigma_dyn=sigma_dyn,
)

# Expose notebook-style names used below
M = rnn.M
N_vecs = rnn.N_vecs
I = rnn.I
J = rnn.J
dim_z = rnn.dim_z

In [ ]:
alpha = 1 - dt / tau
print(alpha)

In [ ]:
# get paramaters in format for scify
A = np.array([(1 - dt / tau), (1 - dt / tau)])
W1 = dt * N_vecs / (tau * N)
W2 = M
h1 = np.zeros(2)
h2 = rnn.unit_bias

lims = 5

# define search grid
n_samples = 100
z_init = np.array(
    [
        np.random.uniform(low=-lims, high=lims, size=n_samples),
        np.random.uniform(low=-lims * 2, high=lims * 2, size=n_samples),
    ]
).T
x_init = z_init @ M.T
D_init = (x_init + h2) > 0

found_lower_orders, found_eigsigma_pats, n_inverses = run_scify(
    np.diag(A),
    W1,
    W2,
    h1,
    h2,
    inner_loop_iterations=10,
    n_inverses_max=1000,
    initial_states=D_init,
)

In [ ]:
evs = np.array(found_eigsigma_pats[0])
print(evs.shape)
Z_fps = np.array(found_lower_orders[0])[:, 0]
print(Z_fps.shape)

In [ ]:
import importlib
import fig_utils.plots as _plots

importlib.reload(_plots)
from fig_utils.plots import compute_ring_subspace_flow, plot_ring_subspace_streamplot

plot_saddles = False
lims = 7
zsp1, zsp2, u_flow, v_flow = compute_ring_subspace_flow(rnn, lims=lims)
plot_ring_subspace_streamplot(
    zsp1,
    zsp2,
    u_flow,
    v_flow,
    Z_fps,
    evs,
    xlabel="subsp. axis 1",
    ylabel="subsp. axis 2",
    cmap=cmap,
    n_stim=P,
    plot_saddles=plot_saddles,
    save_path="../paper_figures/ring_subspace_S1.pdf",
)

In [ ]:
plot_ring_subspace_streamplot(
    zsp1,
    zsp2,
    u_flow,
    v_flow,
    Z_fps,
    evs,
    box_w=0.9,
    box_h=0.9,
    cmap=cmap,
    n_stim=P,
    plot_saddles=plot_saddles,
    save_path="../paper_figures/ring_subspace_F1.pdf",
    s=20,
)

In [ ]:
# Input works: noisy trajectories that converge back after stimulus offset

n_repeats_demo = 12
trial_duration_s = 10.0
stim_onset_s = 0.55
stim_duration_s = 0.25

ring_task_params = {
    "incl_ns_stim": False,
    "incl_probe": False,
    "incl_cue": False,
    "incl_ramp": False,
}
u_master, _, _, _ = make_all_trials(
    ring_task_params,
    dur=trial_duration_s,
    n_stim=P,
    n_pos=1,
    onset=stim_onset_s,
    stim_dur=stim_duration_s,
    bin_size=dt,
    interval_dur="mean",
    delay_dur="mean",
)
u_ring_demo = np.repeat(u_master, n_repeats_demo, axis=0)
labels_p_demo = np.repeat(np.arange(P), n_repeats_demo)
Z0 = np.random.randn(u_ring_demo.shape[0], rnn.dim_z) * input_noise
Z_hist, _ = rnn.simulate(u_ring_demo, Z0=Z0, return_x=False)

plot_ring_demo_trajectories(
    Z_hist, Z_fps, evs, zsp1, zsp2, cmap=cmap, n_stim=P, panel_gap_x=0.5
)

In [ ]:
plot_connectivity_matrix(J, N)

In [ ]:
trial_duration_s = 4.0
n_repeats = 50
P = 6


u_master, _, _, _ = make_all_trials(
    ring_task_params,
    dur=trial_duration_s,
    n_stim=P,
    n_pos=1,
    onset=stim_onset_s,
    stim_dur=stim_duration_s,
    bin_size=dt,
    interval_dur="mean",
    delay_dur="mean",
)
u = np.repeat(u_master, n_repeats, axis=0)
labels_p = np.repeat(np.arange(P), n_repeats)
n_steps = u.shape[-1]
Z, X = rnn.simulate(u, ic_scale=sigma_dyn, return_x=True)
# reshape to legacy shapes used later in this notebook
xs_all = Z.reshape(P, n_repeats, rnn.dim_z, n_steps)
ys_all = X.reshape(P, n_repeats, N, n_steps)

In [ ]:
unit_ids = [
    np.random.choice(
        np.arange(pop_id * N // 4, (pop_id + 1) * N // 4), size=1, replace=False
    )[0]
    for pop_id in range(4)
]
print(unit_ids)

In [ ]:
st_id = 0
plot_unit_trials_overlay(
    ys_all, unit_ids, stim_index=st_id, n_steps=n_steps, panel_gap_y=0.3
)

In [ ]:
import importlib
import fig_utils.plots as _plots

importlib.reload(_plots)
from fig_utils.plots import ring_cmap_index

run_pyvista = True

# xs_mean should be (P, 2, T). If missing, use class means from xs_all.
xs_mean = xs_all.mean(axis=1)  # (P, dim_z, T)

if run_pyvista:

    # Use kappa1/kappa2 as the two latent dims; treat as a single 'position'
    Z_T = xs_mean  # (P, 2, T)
    traj_color_inds = np.array([[ring_cmap_index(k, P)] for k in range(P)])

    _plots.plot_basis_3d_trajectories(
        Z_T * 2,
        traj_color_inds,
        pos_ind=0,
        cmap=cmap,
        n_pcs_time=0,
        bin_size=dt,
        plt_start=0,
        plt_end=n_steps - 30,
        mirror_x=True,
        jupyter_backend="static",
        window_size=(2000, 1400),
        show=True,
        xscale=1,
        yscale=1.5,
        zscale=1,
        tscale=1,
        stim_mark_times=[stim_onset_s],
        tube_radius=0.02,
        azimuth=5,
        elevation=-30,
        zoom=1.2,
        x_axis_scale=0.7,
        y_axis_scale=1,
        z_axis_scale=0.7,
        axis_line_width=8,
        stim_line_width=4,
        shadow_opacity=0.2,
        shadow_line_width=10,
    )

In [ ]:
n_repeats = 100
P = 6
BINS_AFTER_LAST_STIM = 9
N_BINS_FOR_R = 15


u_master, _, _, _ = make_all_trials(
    ring_task_params,
    dur=trial_duration_s,
    n_stim=P,
    n_pos=1,
    cue_dur=-1,
    bin_size=dt,
    interval_dur="mean",
    delay_dur="mean",
)
u = np.repeat(u_master, n_repeats, axis=0)
labels_p = np.repeat(np.arange(P), n_repeats)
n_steps = u.shape[-1]
last_stim_end = int(stim_end_bins(u).max())
t_perturb = last_stim_end + BINS_AFTER_LAST_STIM
t_move_start = t_perturb + 1
t_move_end = t_move_start + N_BINS_FOR_R

# Match notebook 08 usage: x-path generation for both baseline and perturbed
# Use a shared z0; trial-to-trial variability comes from injected noise.
rnn.z0 = np.zeros(rnn.dim_z)

Z_base = generate_w_perturb_x(rnn, u=u, noise_scale=noise_scale)
xs_all = Z_base.reshape(P, n_repeats, rnn.dim_z, n_steps)

z1 = 0
z2 = 1
pert_z1 = perturbation_goal_bounds(Z_base[:, z1, t_perturb])
pert_z2 = perturbation_goal_bounds(Z_base[:, z2, t_perturb])

# Per-trial target goals in z-space (kappa1,kappa2)
goals = np.random.rand(u.shape[0], 2)
goals[:, 0] = goals[:, 0] * (pert_z1[1] - pert_z1[0]) + pert_z1[0]
goals[:, 1] = goals[:, 1] * (pert_z2[1] - pert_z2[0]) + pert_z2[0]

# Map x -> (kappa1,kappa2) for perturbation geometry
pert_dir = rnn.M  # (N, dim_z)
n_pcs_time = 0
pos = 0
perturb_inds = list(position_latent_indices(n_pcs_time, pos))  # [0, 1] for ring

goal_amp = 1.0

Z_pert = generate_w_perturb_x(
    rnn,
    u=u,
    noise_scale=noise_scale,
    perturb_at_t=t_perturb,
    perturb_weights=pert_dir,
    perturb_inds=perturb_inds,
    goal=goals,
    goal_amp=goal_amp,
    optogen=0.0,
)

xs_all_perturb = Z_pert.reshape(P, n_repeats, rnn.dim_z, n_steps)

In [ ]:
goals.shape

In [ ]:
# Use the same 2D plane as elsewhere in this notebook


# Perturbation snapshots (reuse notebook 05/fig_utils style)
# xs_all, xs_all_perturb: (P, n_repeats, dim_z, n_steps)

P, n_repeats, dim_z, n_steps = xs_all.shape

# Use the ring kappa plane
z1 = 0
z2 = 1
Z = xs_all.reshape(-1, dim_z, n_steps)
Z_pert = xs_all_perturb.reshape(-1, dim_z, n_steps)
labels_pos = np.repeat(np.arange(P), n_repeats)

# snapshots around perturbation time
plot_ts = [t_perturb - 2, t_perturb, t_perturb + 10, t_move_end]
highlight_cond = 2

plot_perturbation_latent_snapshots_attractor(
    Z,
    Z_pert,
    labels_pos,
    z1=z1,
    z2=z2,
    cmap=cmap,
    plot_ts=plot_ts,
    highlight_cond=highlight_cond,
    bin_size=dt,
    show=True,
)

In [ ]:
t1 = t_move_start
t2 = t_move_end
z1 = 0
z2 = 1

In [ ]:
# Distance moved vs distance-to-manifold (reuse fig_utils.perturbation)

P, n_repeats, dim_z, n_steps = xs_all.shape
Z = xs_all.reshape(-1, dim_z, n_steps)
Z_pert = xs_all_perturb.reshape(-1, dim_z, n_steps)
labels_pos = np.repeat(np.arange(P), n_repeats)

stats = compute_perturbation_distance_stats(
    Z,
    Z_pert,
    labels_pos,
    z1=z1,
    z2=z2,
    t_move_start=t1,
    t_move_end=t2,
)

In [ ]:
plot_distance_moved_vs_class_mean(
    stats["distances_to_manifolds"],
    stats["distance_moved"],
    slope=stats["slope"],
    intercept=stats["intercept"],
    pearson_r=stats["pearson_r"],
    dpi=300,
    show=True,
    box_w=0.6,
    box_h=0.6,
    # max_x = 30,
    max_y=4,
)

### Generate ground truth data from this system

In [ ]:
# Shared student–teacher trial design (both hand-constructed teachers).
ST_TRIAL_DEFAULTS = {
    "n_trials": 6000,
    "n_conditions": 6,
    "n_sessions": 8,
    "train_perc": 0.75,
    "trial_duration_s": 4.0,
    "stim_onset_s": (0.5, 0.7),
    "stim_duration_s": 0.25,
    "observation_on": "currents",
}

# Observation gains tuned to match recorded marginal spike statistics.
ST_OBS_RING = {"w_obs": 0.1, "bias_scale": 0.5, "bias_mean": -0.1}
# ST_OBS_RING = {"w_obs": .3, "bias_scale": 0.1, "bias_mean": -1}

dataset_kwargs = {**ST_TRIAL_DEFAULTS, **ST_OBS_RING}

dataset_seed = int(np.random.randint(1e6)) if save else seed
st_data = generate_student_teacher_dataset(
    rnn,
    seed=dataset_seed,
    dataset_name="ring_attractor",
    save_path="../data/synthetic" if save else None,
    **dataset_kwargs,
)

y = st_data["y"]
# u = st_data["u"]
trial_labels = st_data["labels"]
xs_all = st_data["currents"]
R_sub = st_data["rates"]
y_rates = st_data["y_rates"]
# unit_biases = st_data["unit_biases"]
# add_params = st_data["add_params"]
task_params = st_data["task_params"]
n_trials = y.shape[0]

In [ ]:
print(dataset_seed)

In [ ]:
y.shape

In [ ]:
# Spike histogram stats
# data stats
# mean population rate (Hz): 15.311
# mean ISI per unit (s): 0.171
# mean |pairwise corr|: 0.036

# subsample random 1000/8 trials and 600/8 units
n_trials_sub = 6000 // 8
n_units_sub = 600 // 8
units = np.random.choice(np.arange(N), size=n_units_sub, replace=False)
trials = np.random.choice(np.arange(n_trials), size=n_trials_sub, replace=False)
y_sub = y[trials][:, units]


stats_synth = spike_histogram_stats_from_array(y_sub, task_params["bin_size"])
plot_spike_histogram_stats(
    stats_synth,
)

In [ ]:
plt.imshow(y[0], aspect="auto")

In [ ]:
plot_student_teacher_overview(
    xs_all,
    R_sub,
    y_rates,
    y,
    trial=0,
    unit_indices=(0, 100),
    box_w=1,
    box_h=0.6,
    panel_gap_x=0.3,
    panel_gap_y=0.5,
)

In [ ]:
plot_unit_activity_by_stimulus(
    st_data=st_data, stimulus=0, panel_gap_y=0.5, box_w=1, box_h=0.6
);